In [1]:
import torch
import numpy as np
from craft import CRAFT
from collections import OrderedDict

In [2]:
craft_model = './weights/craft_mlt_25k.pth'

In [3]:
def copyStateDict(state_dict):
    if list(state_dict.keys())[0].startswith("module"):
        start_idx = 1
    else:
        start_idx = 0
    new_state_dict = OrderedDict()
    for k, v in state_dict.items():
        name = ".".join(k.split(".")[start_idx:])
        new_state_dict[name] = v
    return new_state_dict

In [4]:
net = CRAFT()
net.load_state_dict(copyStateDict(torch.load(craft_model, map_location = 'cpu')))
net.eval()

CRAFT(
  (basenet): vgg16_bn(
    (slice1): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
      (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (7): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (8): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (9): ReLU(inplace=True)
      (10): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (11): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (slice2): Sequential(
      (12): ReLU(inplace=True)
      (13): MaxPool2d(kerne

In [5]:
width = 32,
height = 32,
dynamic_axes = {"input" : {2 : "width", 3 : "height"}} # dynamic axes for the width and height
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [17]:
torch.onnx.export(net,
                  torch.randn(1,3,32,32).to(device), #indexes with 0, 1, 2, 3
                  "./weights/craft.onnx",
                  opset_version = 11,
                  input_names = ['input'],
                  output_names = ['output1','output2'],
                  dynamic_axes = dynamic_axes)